# 💎 AudioX Studio (Community Edition)
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

Welcome to the advanced AudioX Colab engine. This community notebook acts as a precision sound design station powered by the native inference API. It features professional audio mixing, allowing you to target exact timestamps to score specific scenes.

In [ ]:
# @title Create Isolated Environment & Install Dependencies
import os

!apt-get update -yq
!apt-get install python3.10 python3.10-venv python3.10-dev -yq
!python3.10 -m venv /content/audiox_env

if not os.path.exists('/content/AudioX'):
    !git clone https://github.com/ZeyueT/AudioX.git

%cd /content/AudioX

!sed -i 's/map_location="cpu"/map_location="cuda"/g' /content/AudioX/audiox/models/utils.py
!sed -i "s/map_location='cpu'/map_location='cuda'/g" /content/AudioX/audiox/models/utils.py

!apt-get install -yq ffmpeg libsndfile1

!/content/audiox_env/bin/python -m pip install --upgrade pip -q
!/content/audiox_env/bin/python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!/content/audiox_env/bin/python -m pip install -e .
!/content/audiox_env/bin/python -m pip install "huggingface_hub<0.27.0" "pydantic<2.9.0" "transformers<5.0.0" omegaconf timm

print("✅ Isolated Setup Complete!")

In [ ]:
# @title Mode 1: 🎛️ Single Scene Precision Scorer
# @markdown Extract, score, and output a single specific scene from your video.
prompt = "Table tennis sound " # @param {type:"string"}
video_path = "/content/sample_data/199833-911378640_tiny (1).mp4" # @param {type:"string"}
start_time_seconds = 0 # @param {type:"number"}
keep_original_audio = False # @param {type:"boolean"}
audio_mix_volume = 0.8 # @param {type:"slider", min:0.1, max:2.0, step:0.1}
cfg_scale = 1 # @param {type:"slider", min:1.0, max:15.0, step:0.5}
steps = 100 # @param {type:"slider", min:10, max:200, step:1}
seed = -1 # @param {type:"integer"}
model_name = "AudioX-MAF-MMDiT" # @param ["AudioX", "AudioX-MAF", "AudioX-MAF-MMDiT"]

import json
import os
import subprocess
from IPython.display import Audio, Video, display
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

# 🧹 Nuke the Linux Page Cache to free "ghost RAM"
os.system('sync && echo 3 > /proc/sys/vm/drop_caches')

# ⚙️ OS Protocol: Swap file safety net
if not os.path.exists('/swapfile'):
    print("⚙️ Allocating 16GB Virtual RAM...", flush=True)
    os.system('fallocate -l 16G /swapfile')
    os.system('chmod 600 /swapfile')
    os.system('mkswap /swapfile')
    os.system('swapon /swapfile')

MODEL_CONFIGS = {
    "AudioX": {"repo": "HKUSTAudio/AudioX"},
    "AudioX-MAF": {"repo": "HKUSTAudio/AudioX-MAF"},
    "AudioX-MAF-MMDiT": {"repo": "HKUSTAudio/AudioX-MAF-MMDiT"}
}

repo_id = f"HKUSTAudio/{model_name}"
print(f"📥 Verifying & Caching {model_name} checkpoints natively...")
hf_hub_download(repo_id=repo_id, filename="config.json")
hf_hub_download(repo_id=repo_id, filename="model.ckpt")

if "MAF" in model_name:
    print(f"👁️ Caching Motion-Aware Synchformer...")
    hf_hub_download(repo_id="HKUSTAudio/AudioX-MAF", filename="synchformer_state_dict.pth")

if model_name in ["AudioX-MAF", "AudioX-MAF-MMDiT"]:
    print(f"🧠 Aligning VAE with researcher architecture...")
    os.makedirs("/content/AudioX/model", exist_ok=True)
    hf_hub_download(repo_id="HKUSTAudio/AudioX-MAF-MMDiT", filename="VAE.ckpt", local_dir="/content/AudioX/model")

# 🧠 Architecture Patch: Fix Researcher's Sloppy Float/Int Type Error
utils_file = "/content/AudioX/audiox/data/utils.py"
if os.path.exists(utils_file):
    with open(utils_file, "r") as f:
        utils_content = f.read()
    utils_content = utils_content.replace("last_frame.repeat(repeat_times,", "last_frame.repeat(int(repeat_times),")
    with open(utils_file, "w") as f:
        f.write(utils_content)

print("✅ Models synchronized and cached successfully!\n")

engine_code = r"""
import os, sys, json, subprocess, tempfile, shutil, inspect, torch, torchaudio, gc
from einops import rearrange
import numpy as np
import soundfile as sf
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

# 🛡️ THE VRAM OFFLOADER (Universal Mmap Edition)
_orig_torch_load = torch.load
def _mock_torch_load(*args, **kwargs):
    kwargs['map_location'] = 'cpu'
    try:
        kwargs['mmap'] = True
        loaded = _orig_torch_load(*args, **kwargs)
    except Exception:
        kwargs.pop('mmap', None)
        loaded = _orig_torch_load(*args, **kwargs)

    if isinstance(loaded, dict) and 'state_dict' in loaded:
        clean_state = loaded['state_dict']
        del loaded
        gc.collect()
        return {'state_dict': clean_state}
    return loaded
torch.load = _mock_torch_load

from audiox import get_pretrained_model
from audiox.inference.generation import generate_diffusion_cond
from audiox.data.utils import read_video, encode_video_with_synchformer

def main():
    with open(sys.argv[1], 'r') as f: args = json.load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = args.get('model_name', 'AudioX-MAF-MMDiT')
    repo_id = f"HKUSTAudio/{model_name}"

    prompt = args['prompt']
    video_path = args.get('video_path')
    start_time = float(args.get('start_time_seconds', 0))
    duration = 10.0

    keep_audio = args.get('keep_original_audio', False)
    mix_vol = args.get('audio_mix_volume', 1.0)

    if ("MAF" in model_name) and (not video_path or not os.path.exists(video_path)):
        print("\n❌ CRITICAL ERROR: Valid 'video_path' required.")
        sys.exit(1)

    print("\n⏳ Fetching configuration...", flush=True)
    config_path_json = hf_hub_download(repo_id=repo_id, filename="config.json")
    with open(config_path_json, 'r') as f:
        model_config = json.load(f)

    sample_rate = model_config.get("sample_rate", 16000)
    sample_size = model_config.get("sample_size", 65536)
    target_fps = model_config.get("video_fps", 8)

    video_tensor = None
    video_sync_frames = None

    if video_path and os.path.exists(video_path):
        print(f"🎯 Slicing video chunk to protect CPU RAM...", flush=True)
        tmp_chunk = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
        # Pre-trim the video with FFmpeg before Python even sees it
        subprocess.run(["ffmpeg", "-y", "-ss", str(start_time), "-i", video_path, "-t", str(duration), "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23", "-an", tmp_chunk], check=True, capture_output=True)

        print(f"🎯 Extracting pre-computed motion features...", flush=True)
        video_tensor = read_video(tmp_chunk, seek_time=0, duration=duration, target_fps=target_fps)
        gc.collect()

        if "MAF" in model_name:
            video_sync_frames = encode_video_with_synchformer(tmp_chunk, "HKUSTAudio/AudioX-MAF", 0, duration, device)

        os.remove(tmp_chunk)

        if keep_audio:
            try:
                audio_streams = subprocess.check_output(["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "csv=p=0", video_path]).decode().strip()
                if not audio_streams: keep_audio = False
            except Exception: keep_audio = False

    gc.collect()
    torch.cuda.empty_cache()

    print("⏳ Loading Main Neural Networks into VRAM (Applying RAM Bypass)...", flush=True)
    torch.set_default_dtype(torch.float16)
    model, _ = get_pretrained_model(repo_id)
    model = model.to(device)
    torch.set_default_dtype(torch.float32)

    audio_tensor = torch.zeros((2, int(sample_rate * duration)))
    video_prompt = {"video_tensors": video_tensor.unsqueeze(0), "video_sync_frames": video_sync_frames} if video_tensor is not None else None

    conditioning = [{
        "video_prompt": video_prompt,
        "text_prompt": prompt,
        "audio_prompt": audio_tensor.unsqueeze(0),
        "seconds_start": start_time,
        "seconds_total": duration
    }]

    print("🚀 Generating Cinematic Score (This may take a minute)...", flush=True)
    with torch.autocast("cuda", dtype=torch.float16):
        output = generate_diffusion_cond(
            model, steps=int(args['steps']), cfg_scale=float(args['cfg_scale']),
            conditioning=conditioning, sample_size=sample_size, sigma_min=0.3,
            sigma_max=500, sampler_type="dpmpp-3m-sde", device=device
        )

    output = rearrange(output, "b d n -> d (b n)")
    output = output.to(torch.float32)
    max_val = torch.max(torch.abs(output))
    if max_val > 0: output = output.div(max_val)
    output = output.clamp(-1, 1).mul(32767).to(torch.int16).cpu()

    save_dir = '/content/AudioX_Outputs'
    os.makedirs(save_dir, exist_ok=True)
    audio_path = tempfile.NamedTemporaryFile(suffix='.wav', delete=False).name
    torchaudio.save(audio_path, output, sample_rate)

    synced_vid = None
    if video_path and os.path.exists(video_path) and video_tensor is not None:
        output_mp4 = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
        extracted_vid_path = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
        subprocess.run(["ffmpeg", "-y", "-ss", str(start_time), "-i", video_path, "-t", str(duration), "-c:v", "libx264", "-preset", "ultrafast", "-c:a", "copy", extracted_vid_path], check=True, capture_output=True)
        try:
            actual_dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", extracted_vid_path]).strip())
            if keep_audio:
                print("🎛️ Mastering audio channels (Merging original dialogue + AI score)...", flush=True)
                filter_complex = f"[1:a]volume={mix_vol}[a1];[0:a][a1]amix=inputs=2:duration=first:dropout_transition=2[a]"
                subprocess.run(["ffmpeg", "-y", "-i", extracted_vid_path, "-i", audio_path, "-filter_complex", filter_complex, "-map", "0:v:0", "-map", "[a]", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-t", str(actual_dur), output_mp4], check=True, capture_output=True)
            else:
                subprocess.run(["ffmpeg", "-y", "-i", extracted_vid_path, "-i", audio_path, "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-map", "0:v:0", "-map", "1:a:0", "-t", str(actual_dur), output_mp4], check=True, capture_output=True)

            base_name = os.path.basename(video_path).split('.')[0]
            synced_vid = os.path.join(save_dir, f"{base_name}_scene_{int(start_time)}s_synced.mp4")
            shutil.copy(output_mp4, synced_vid)
        except Exception: pass

    safe_audio_path = os.path.join(save_dir, f"AudioX_Premium_{np.random.randint(10000)}.wav")
    shutil.copy(audio_path, safe_audio_path)

    print("\n--- 💎 SCENE GENERATION COMPLETE ---", flush=True)
    print(f"AUDIO_PATH:{safe_audio_path}", flush=True)
    if synced_vid: print(f"VIDEO_PATH:{synced_vid}", flush=True)

    with open('/content/AudioX/output_paths.json', 'w') as f:
        json.dump({'audio_out': safe_audio_path, 'video_out': synced_vid}, f)

if __name__ == "__main__": main()
"""

with open("/content/AudioX/run_single_engine.py", "w") as f:
    f.write(engine_code)

config = {
    "prompt": prompt,
    "video_path": video_path if video_path.strip() != "" else None,
    "start_time_seconds": start_time_seconds,
    "keep_original_audio": keep_original_audio,
    "audio_mix_volume": audio_mix_volume,
    "cfg_scale": cfg_scale,
    "steps": steps,
    "seed": seed,
    "model_name": model_name
}

config_path = "/content/AudioX/single_config.json"
with open(config_path, "w") as f:
    json.dump(config, f)

if os.path.exists("/content/AudioX/output_paths.json"):
    os.remove("/content/AudioX/output_paths.json")

print("🚀 Firing up AudioX Precision Engine...\n")
process = subprocess.Popen(
    ["/content/audiox_env/bin/python", "/content/AudioX/run_single_engine.py", config_path],
    cwd="/content/AudioX",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    print(f"\n❌ FATAL CRASH: Engine exited with code {process.returncode}")
else:
    audio_out = None
    video_out = None
    if os.path.exists("/content/AudioX/output_paths.json"):
        with open("/content/AudioX/output_paths.json", "r") as f:
            out_data = json.load(f)
            audio_out = out_data.get("audio_out")
            video_out = out_data.get("video_out")

    print("\n✨ Rendering outputs...\n")
    if video_out and os.path.exists(video_out):
        display(Video(video_out, embed=True, html_attributes="controls width='600'"))
    elif audio_out and os.path.exists(audio_out):
        display(Audio(audio_out))

---
## 💖 Enjoying this tool? Support the Channel!

This **Community Edition** gives you full access to the Mode 1 Single Scene Precision Scorer. I hope it helps you out in your creative workflows!

If you find my tutorials helpful and want to support the **AI With Chucky** YouTube channel, consider joining our Patreon family. Your support is what keeps these free tools and projects alive.

As a massive *thank you* to my supporters, I've included a few extra perks in the full version of this notebook over on Patreon:
*   **Mode 2: 🎬 Full-Timeline Multi-Scene Foley** (Score an entire video by specifying multiple exact time codes and prompts at once!)
*   **Mode 3: 📁 Premium Batch Processor** (Process entire folders of clips automatically to a downloadable Zip)

👉 **[Support the channel on Patreon & check out the extra features here!](https://www.patreon.com/posts/add-audio-to-ai-151467874)**

*Thank you for watching, creating, and being part of the community!*